In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df=pd.read_json('/content/drive/MyDrive/QA data /Langgraph/positive_pairs.jsonl', lines=True)
df.head()

,document_id,anchor,positive,framework,source
0,doc_a20bc082c284ba11cedb556673e340645468f8a1,How does test_entrypoint_output_schema_with_re...,def test_entrypoint_output_schema_with_return_...,unknown,synthetic_positive_v2
1,doc_a20bc082c284ba11cedb556673e340645468f8a1,Best practices for test_entrypoint_output_sche...,def test_entrypoint_output_schema_with_return_...,unknown,synthetic_positive_v2
2,doc_a20bc082c284ba11cedb556673e340645468f8a1,Example usage of test_entrypoint_output_schema...,def test_entrypoint_output_schema_with_return_...,unknown,synthetic_positive_v2
3,doc_a20bc082c284ba11cedb556673e340645468f8a1,Explain the test_entrypoint_output_schema_with...,def test_entrypoint_output_schema_with_return_...,unknown,synthetic_positive_v2
4,doc_a20bc082c284ba11cedb556673e340645468f8a1,How to implement test_entrypoint_output_schema...,def test_entrypoint_output_schema_with_return_...,unknown,synthetic_positive_v2


In [ ]:
## drop column
df.drop(['framework','source'], axis=1, inplace=True)

In [ ]:
## adding a column
df['id'] = range(len(df))
df.head(5)

,document_id,anchor,positive,id
0,doc_a20bc082c284ba11cedb556673e340645468f8a1,How does test_entrypoint_output_schema_with_re...,def test_entrypoint_output_schema_with_return_...,0
1,doc_a20bc082c284ba11cedb556673e340645468f8a1,Best practices for test_entrypoint_output_sche...,def test_entrypoint_output_schema_with_return_...,1
2,doc_a20bc082c284ba11cedb556673e340645468f8a1,Example usage of test_entrypoint_output_schema...,def test_entrypoint_output_schema_with_return_...,2
3,doc_a20bc082c284ba11cedb556673e340645468f8a1,Explain the test_entrypoint_output_schema_with...,def test_entrypoint_output_schema_with_return_...,3
4,doc_a20bc082c284ba11cedb556673e340645468f8a1,How to implement test_entrypoint_output_schema...,def test_entrypoint_output_schema_with_return_...,4


In [ ]:
unique_document_ids = df['document_id'].unique()
id_to_numeric = {doc_id: i for i, doc_id in enumerate(unique_document_ids)}
df['document_id'] = df['document_id'].map(id_to_numeric)
df.head()

,document_id,anchor,positive,id
0,0,How does test_entrypoint_output_schema_with_re...,def test_entrypoint_output_schema_with_return_...,0
1,0,Best practices for test_entrypoint_output_sche...,def test_entrypoint_output_schema_with_return_...,1
2,0,Example usage of test_entrypoint_output_schema...,def test_entrypoint_output_schema_with_return_...,2
3,0,Explain the test_entrypoint_output_schema_with...,def test_entrypoint_output_schema_with_return_...,3
4,0,How to implement test_entrypoint_output_schema...,def test_entrypoint_output_schema_with_return_...,4


In [ ]:
## shuffle the df
df = df.sample(frac=1).reset_index(drop=True)
df.head(5)

,document_id,anchor,positive,id
0,121,Example usage of put_writes,"def put_writes(\n self,\n config...",609
1,168,How does async __aenter__ work in Python?,async def __aenter__(self) -> InMemorySaver:\n...,844
2,178,Example usage of InputState,class InputState(TypedDict):\n question...,893
3,40,Example usage of clear_cache,"def clear_cache(self, nodes: Sequence[str] | N...",200
4,52,Example usage of __eq__,"def __eq__(self, other: object) -> bool:\n ...",260


In [ ]:
# Split Df Into a 90/10 Train/Test split using scikit-learn train test split
from sklearn.model_selection import train_test_split
train, test = train_test_split(df, test_size=0.1)

In [ ]:
## save train and test to json
train.to_json('train.json', orient='records', lines=True)
test.to_json('test.json', orient='records', lines=True)

In [ ]:
# Convert datasets into dictionary format required by the InformationRetrievalEvaluator

# corpus: maps corpus IDs to their text chunks (documents)
# Format: {corpus_id: text_chunk}
corpus = df.set_index('id')['positive'].to_dict()

# queries: maps query IDs to their questions
# Format: {query_id: question_text}
queries = test.set_index('id')['anchor'].to_dict()

# Create a mapping between queries and their relevant documents
# This tells the evaluator which documents are correct matches for each query
relevant_docs = {}
for q_id, document_id in zip(test["id"], test["document_id"]):
    # Initialize empty list for each query if not already present
    if q_id not in relevant_docs:
        relevant_docs[q_id] = []

    # Find all corpus entries that share the same global_chunk_id
    # This handles cases where multiple questions can refer to the same text chunk
    matching_corpus_ids = [
        cid for cid, chunk in zip(df["id"], df["document_id"])
        if chunk == document_id
    ]
    # Add the matching corpus IDs to the relevant documents for this query
    relevant_docs[q_id].extend(matching_corpus_ids)

## Baseline Evaluation

#### base codeBERT

In [ ]:
import torch

from sentence_transformers import SentenceTransformer, SentenceTransformerModelCardData, SentenceTransformerTrainingArguments, SentenceTransformerTrainer
from sentence_transformers.evaluation import InformationRetrievalEvaluator, SequentialEvaluator
from sentence_transformers.util import cos_sim
from sentence_transformers.losses import MatryoshkaLoss, MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

In [ ]:
# Hugging Face model ID
model_id = "microsoft/codebert-base"

# Loading via SentenceTransformer
model = SentenceTransformer(
    model_id, device="cuda" if torch.cuda.is_available() else "cpu"
)

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [ ]:
# Dimensions of interest
matryoshka_dimensions = [768, 512, 256, 128, 64] # Important: large to small

# Create empty list to hold evaluators
matryoshka_evaluators = []

# Create an evaluator for each above dimension
for dim in matryoshka_dimensions:
    # Define the evaluator
    ir_evaluator = InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        name=f"dim_{dim}",
        truncate_dim=dim,  # Truncate the embeddings to the respective dimension
        score_functions={"cosine": cos_sim},
    )
    # Add to list
    matryoshka_evaluators.append(ir_evaluator)

# Create a sequential evaluator
# Able to run all our dimension specific InformationRetrievalEvaluators sequentially.
evaluator = SequentialEvaluator(matryoshka_evaluators)


In [ ]:
# Evaluate the model
base_results = evaluator(model)

# Print header
print("\nBase Model Evaluation Results")
print("-" * 85)
print(f"{'Metric':15} {'768d':>12} {'512d':>12} {'256d':>12} {'128d':>12} {'64d':>12}")
print("-" * 85)

# List of metrics to display
metrics = [
    'ndcg@10',
    'mrr@10',
    'accuracy@1',
    'precision@3',
    'precision@5',
    'recall@3',

]

# Print each metric
for metric in metrics:
    values = []
    for dim in matryoshka_dimensions:
        key = f"dim_{dim}_cosine_{metric}"
        values.append(base_results[key])

    # Highlight NDCG@10
    metric_name = f"=={metric}==" if metric == "ndcg@10" else metric
    print(f"{metric_name:15}", end="  ")
    for val in values:
        print(f"{val:12.4f}", end=" ")
    print()

# Print sequential score
print("-" * 85)
print(f"{'seq_score:'} {base_results['sequential_score']:1f}")


Base Model Evaluation Results
-------------------------------------------------------------------------------------
Metric                  768d         512d         256d         128d          64d
-------------------------------------------------------------------------------------
==ndcg@10==            0.0554       0.0754       0.0700       0.0600       0.0700 
mrr@10                 0.0517       0.0717       0.0700       0.0600       0.0700 
accuracy@1             0.0500       0.0700       0.0700       0.0600       0.0700 
precision@3            0.0500       0.0700       0.0700       0.0600       0.0700 
precision@5            0.0500       0.0700       0.0700       0.0600       0.0700 
recall@3               0.0300       0.0420       0.0420       0.0360       0.0420 
-------------------------------------------------------------------------------------
seq_score: 0.070000


### already finetuned model on langchain data

In [ ]:
# Hugging Face model ID
model_id = "shubharuidas/codebert-embed-base-dense-retriever"

# Loading via SentenceTransformer
model = SentenceTransformer(
    model_id, device="cuda" if torch.cuda.is_available() else "cpu"
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/283 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/312 [00:00<?, ?B/s]

In [ ]:
# Evaluate the model
base_results = evaluator(model)

# Print header
print("\nBase Model Evaluation Results")
print("-" * 85)
print(f"{'Metric':15} {'768d':>12} {'512d':>12} {'256d':>12} {'128d':>12} {'64d':>12}")
print("-" * 85)

# List of metrics to display
metrics = [
    'ndcg@10',
    'mrr@10',
    'accuracy@1',
    'precision@3',
    'precision@5',
    'recall@3',

]

# Print each metric
for metric in metrics:
    values = []
    for dim in matryoshka_dimensions:
        key = f"dim_{dim}_cosine_{metric}"
        values.append(base_results[key])

    # Highlight NDCG@10
    metric_name = f"=={metric}==" if metric == "ndcg@10" else metric
    print(f"{metric_name:15}", end="  ")
    for val in values:
        print(f"{val:12.4f}", end=" ")
    print()

# Print sequential score
print("-" * 85)
print(f"{'seq_score:'} {base_results['sequential_score']:1f}")


Base Model Evaluation Results
-------------------------------------------------------------------------------------
Metric                  768d         512d         256d         128d          64d
-------------------------------------------------------------------------------------
==ndcg@10==            0.8225       0.8270       0.8370       0.8241       0.7387 
mrr@10                 0.8000       0.8083       0.8183       0.7867       0.7050 
accuracy@1             0.7900       0.8000       0.8100       0.7700       0.6900 
precision@3            0.7900       0.8000       0.8100       0.7700       0.6900 
precision@5            0.7900       0.8000       0.8100       0.7700       0.6900 
recall@3               0.4740       0.4800       0.4860       0.4620       0.4140 
-------------------------------------------------------------------------------------
seq_score: 0.738690


# Fine Tuning codeBERT from scratch on langgraph

In [ ]:
# Hugging Face model ID
model_id = "microsoft/codebert-base"

# Loading via SentenceTransformer
model = SentenceTransformer(
    model_id, device="cuda" if torch.cuda.is_available() else "cpu"
)

In [ ]:
from sentence_transformers.datasets import ParallelSentencesDataset
from sentence_transformers.losses import CosineSimilarityLoss

In [ ]:
# load model with SDPA for using Flash Attention 2
model = SentenceTransformer(
    model_id,
    model_kwargs={"attn_implementation": "sdpa"},
    model_card_data=SentenceTransformerModelCardData(
        language="en",
        license="apache-2.0",
        model_name="codeBert Base",
    ),
)

In [ ]:
# Initial Loss
base_loss = MultipleNegativesRankingLoss(model)

# Matryoshka Loss Wrapper
train_loss = MatryoshkaLoss(
    model, base_loss, matryoshka_dims=matryoshka_dimensions
)

In [ ]:
args = SentenceTransformerTrainingArguments(
    output_dir="codebert-base-shubha",
    num_train_epochs=4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=16,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    fp16=True,
    bf16=False,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_dim_128_cosine_ndcg@10",
    greater_is_better=True,
    report_to="none",
)


In [ ]:
from datasets import Dataset

# Explicitly keep ONLY text columns
train_dataset = Dataset.from_pandas(
    train[["anchor", "positive"]].astype(str),
    preserve_index=False
)


In [ ]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,  # training dataset
    loss=train_loss,
    evaluator=evaluator,
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

In [ ]:
# Start training
trainer.train()

Epoch,Training Loss,Validation Loss,Dim 768 Cosine Accuracy@1,Dim 768 Cosine Accuracy@3,Dim 768 Cosine Accuracy@5,Dim 768 Cosine Accuracy@10,Dim 768 Cosine Precision@1,Dim 768 Cosine Precision@3,Dim 768 Cosine Precision@5,Dim 768 Cosine Precision@10,Dim 768 Cosine Recall@1,Dim 768 Cosine Recall@3,Dim 768 Cosine Recall@5,Dim 768 Cosine Recall@10,Dim 768 Cosine Ndcg@10,Dim 768 Cosine Mrr@10,Dim 768 Cosine Map@100,Dim 512 Cosine Accuracy@1,Dim 512 Cosine Accuracy@3,Dim 512 Cosine Accuracy@5,Dim 512 Cosine Accuracy@10,Dim 512 Cosine Precision@1,Dim 512 Cosine Precision@3,Dim 512 Cosine Precision@5,Dim 512 Cosine Precision@10,Dim 512 Cosine Recall@1,Dim 512 Cosine Recall@3,Dim 512 Cosine Recall@5,Dim 512 Cosine Recall@10,Dim 512 Cosine Ndcg@10,Dim 512 Cosine Mrr@10,Dim 512 Cosine Map@100,Dim 256 Cosine Accuracy@1,Dim 256 Cosine Accuracy@3,Dim 256 Cosine Accuracy@5,Dim 256 Cosine Accuracy@10,Dim 256 Cosine Precision@1,Dim 256 Cosine Precision@3,Dim 256 Cosine Precision@5,Dim 256 Cosine Precision@10,Dim 256 Cosine Recall@1,Dim 256 Cosine Recall@3,Dim 256 Cosine Recall@5,Dim 256 Cosine Recall@10,Dim 256 Cosine Ndcg@10,Dim 256 Cosine Mrr@10,Dim 256 Cosine Map@100,Dim 128 Cosine Accuracy@1,Dim 128 Cosine Accuracy@3,Dim 128 Cosine Accuracy@5,Dim 128 Cosine Accuracy@10,Dim 128 Cosine Precision@1,Dim 128 Cosine Precision@3,Dim 128 Cosine Precision@5,Dim 128 Cosine Precision@10,Dim 128 Cosine Recall@1,Dim 128 Cosine Recall@3,Dim 128 Cosine Recall@5,Dim 128 Cosine Recall@10,Dim 128 Cosine Ndcg@10,Dim 128 Cosine Mrr@10,Dim 128 Cosine Map@100,Dim 64 Cosine Accuracy@1,Dim 64 Cosine Accuracy@3,Dim 64 Cosine Accuracy@5,Dim 64 Cosine Accuracy@10,Dim 64 Cosine Precision@1,Dim 64 Cosine Precision@3,Dim 64 Cosine Precision@5,Dim 64 Cosine Precision@10,Dim 64 Cosine Recall@1,Dim 64 Cosine Recall@3,Dim 64 Cosine Recall@5,Dim 64 Cosine Recall@10,Dim 64 Cosine Ndcg@10,Dim 64 Cosine Mrr@10,Dim 64 Cosine Map@100,Sequential Score
1,7.080400,No log,0.100000,0.100000,0.100000,0.110000,0.100000,0.100000,0.100000,0.055000,0.020000,0.060000,0.100000,0.110000,0.105410,0.101667,0.125774,0.050000,0.050000,0.050000,0.060000,0.050000,0.050000,0.050000,0.030000,0.010000,0.030000,0.050000,0.060000,0.055410,0.051667,0.072255,0.040000,0.040000,0.040000,0.050000,0.040000,0.040000,0.040000,0.025000,0.008000,0.024000,0.040000,0.050000,0.045410,0.041667,0.064552,0.070000,0.070000,0.070000,0.090000,0.070000,0.070000,0.070000,0.045000,0.014000,0.042000,0.070000,0.090000,0.080820,0.073333,0.090594,0.070000,0.070000,0.070000,0.100000,0.070000,0.070000,0.070000,0.050000,0.014000,0.042000,0.070000,0.100000,0.086230,0.075000,0.108492,0.086230
2,4.556300,No log,0.150000,0.150000,0.150000,0.230000,0.150000,0.150000,0.150000,0.115000,0.030000,0.090000,0.150000,0.230000,0.193280,0.163333,0.206913,0.120000,0.120000,0.120000,0.180000,0.120000,0.120000,0.120000,0.090000,0.024000,0.072000,0.120000,0.180000,0.152460,0.130000,0.172898,0.120000,0.120000,0.120000,0.200000,0.120000,0.120000,0.120000,0.100000,0.024000,0.072000,0.120000,0.200000,0.163280,0.133333,0.176174,0.130000,0.130000,0.130000,0.220000,0.130000,0.130000,0.130000,0.110000,0.026000,0.078000,0.130000,0.220000,0.178690,0.145000,0.188206,0.160000,0.160000,0.160000,0.220000,0.160000,0.160000,0.160000,0.110000,0.032000,0.096000,0.160000,0.220000,0.192460,0.170000,0.218416,0.192460
3,3.329100,No log,0.360000,0.360000,0.360000,0.440000,0.360000,0.360000,0.360000,0.220000,0.072000,0.216000,0.360000,0.440000,0.403280,0.373333,0.431452,0.370000,0.370000,0.370000,0.450000,0.370000,0.370000,0.370000,0.225000,0.074000,0.222000,0.370000,0.450000,0.413280,0.383333,0.437527,0.350000,0.350000,0.350000,0.420000,0.350000,0.350000,0.350000,0.210000,0.070000,0.210000,0.350000,0.420000,0.387870,0.361667,0.415232,0.310000,0.310000,0.310000,0.400000,0.310000,0.310000,0.310000,0.200000,0.062000,0.186000,0.310000,0.400000,0.358690,0.325000,0.386581,0.340000,0.340000,0.340000,0.430000,0.340000,0.340000,0.340000,0.215000,0.068000,0.204000,0.340000,0.4300

TrainOutput(global_step=60, training_loss=4.240740585327148, metrics={'train_runtime': 398.1092, 'train_samples_per_second': 9.043, 'train_steps_per_second': 0.151, 'total_flos': 0.0, 'train_loss': 4.240740585327148, 'epoch': 4.0})

In [ ]:
# Save the best model based on our eval_dim_128_cosine_ndcg@10 criteria
trainer.save_model()

In [ ]:
fine_tuned_model = SentenceTransformer(
    args.output_dir, device="cuda" if torch.cuda.is_available() else "cpu"
)

# Evaluate the model
ft_results = evaluator(fine_tuned_model)

# Print header
print("Fine Tuned Model Evaluation Results")
print("-" * 85)
print(f"{'Metric':15} {'768d':>12} {'512d':>12} {'256d':>12} {'128d':>12} {'64d':>12}")
print("-" * 85)

# List of metrics to display
metrics = [
    'ndcg@10',
    'mrr@10',
    'accuracy@1',
    'precision@3',
    'precision@5',
    'recall@3',

]
# Print each metric
for metric in metrics:
    values = []
    for dim in matryoshka_dimensions:
        key = f"dim_{dim}_cosine_{metric}"
        values.append(ft_results[key])

    # Highlight NDCG@10
    metric_name = f"=={metric}==" if metric == "ndcg@10" else metric
    print(f"{metric_name:15}", end="  ")
    for val in values:
        print(f"{val:12.4f}", end=" ")
    print()

# Print sequential score
print("-" * 85)
print(f"{'seq_score:'} {ft_results['sequential_score']:1f}")

Fine Tuned Model Evaluation Results
-------------------------------------------------------------------------------------
Metric                  768d         512d         256d         128d          64d
-------------------------------------------------------------------------------------
==ndcg@10==            0.4895       0.5041       0.4879       0.4787       0.4495 
mrr@10                 0.4483       0.4667       0.4617       0.4450       0.4083 
accuracy@1             0.4300       0.4500       0.4500       0.4300       0.3900 
precision@3            0.4300       0.4500       0.4500       0.4300       0.3900 
precision@5            0.4300       0.4500       0.4500       0.4300       0.3900 
recall@3               0.2580       0.2700       0.2700       0.2580       0.2340 
-------------------------------------------------------------------------------------
seq_score: 0.449509


# Fine Tuning already langchain trained model

In [ ]:
# Hugging Face model ID
model_id = "shubharuidas/codebert-embed-base-dense-retriever"

# Loading via SentenceTransformer
model = SentenceTransformer(
    model_id, device="cuda" if torch.cuda.is_available() else "cpu"
)

In [ ]:
# load model with SDPA for using Flash Attention 2
model = SentenceTransformer(
    model_id,
    model_kwargs={"attn_implementation": "sdpa"},
    model_card_data=SentenceTransformerModelCardData(
        language="en",
        license="apache-2.0",
        model_name="codeBert dense retriever",
    ),
)

In [ ]:
# Initial Loss
base_loss = MultipleNegativesRankingLoss(model)

# Matryoshka Loss Wrapper
train_loss = MatryoshkaLoss(
    model, base_loss, matryoshka_dims=matryoshka_dimensions
)

In [ ]:
args1 = SentenceTransformerTrainingArguments(
    output_dir="codebert-dense-retriever-shubha",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=16,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    fp16=True,
    bf16=False,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_dim_128_cosine_ndcg@10",
    greater_is_better=True,
    report_to="none",
)


In [ ]:
from datasets import Dataset

# Explicitly keep ONLY text columns
train_dataset = Dataset.from_pandas(
    train[["anchor", "positive"]].astype(str),
    preserve_index=False
)


In [ ]:
trainer1 = SentenceTransformerTrainer(
    model=model,
    args=args1,
    train_dataset=train_dataset,  # training dataset
    loss=train_loss,
    evaluator=evaluator,
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

In [ ]:
# Start training
trainer1.train()

Epoch,Training Loss,Validation Loss,Dim 768 Cosine Accuracy@1,Dim 768 Cosine Accuracy@3,Dim 768 Cosine Accuracy@5,Dim 768 Cosine Accuracy@10,Dim 768 Cosine Precision@1,Dim 768 Cosine Precision@3,Dim 768 Cosine Precision@5,Dim 768 Cosine Precision@10,Dim 768 Cosine Recall@1,Dim 768 Cosine Recall@3,Dim 768 Cosine Recall@5,Dim 768 Cosine Recall@10,Dim 768 Cosine Ndcg@10,Dim 768 Cosine Mrr@10,Dim 768 Cosine Map@100,Dim 512 Cosine Accuracy@1,Dim 512 Cosine Accuracy@3,Dim 512 Cosine Accuracy@5,Dim 512 Cosine Accuracy@10,Dim 512 Cosine Precision@1,Dim 512 Cosine Precision@3,Dim 512 Cosine Precision@5,Dim 512 Cosine Precision@10,Dim 512 Cosine Recall@1,Dim 512 Cosine Recall@3,Dim 512 Cosine Recall@5,Dim 512 Cosine Recall@10,Dim 512 Cosine Ndcg@10,Dim 512 Cosine Mrr@10,Dim 512 Cosine Map@100,Dim 256 Cosine Accuracy@1,Dim 256 Cosine Accuracy@3,Dim 256 Cosine Accuracy@5,Dim 256 Cosine Accuracy@10,Dim 256 Cosine Precision@1,Dim 256 Cosine Precision@3,Dim 256 Cosine Precision@5,Dim 256 Cosine Precision@10,Dim 256 Cosine Recall@1,Dim 256 Cosine Recall@3,Dim 256 Cosine Recall@5,Dim 256 Cosine Recall@10,Dim 256 Cosine Ndcg@10,Dim 256 Cosine Mrr@10,Dim 256 Cosine Map@100,Dim 128 Cosine Accuracy@1,Dim 128 Cosine Accuracy@3,Dim 128 Cosine Accuracy@5,Dim 128 Cosine Accuracy@10,Dim 128 Cosine Precision@1,Dim 128 Cosine Precision@3,Dim 128 Cosine Precision@5,Dim 128 Cosine Precision@10,Dim 128 Cosine Recall@1,Dim 128 Cosine Recall@3,Dim 128 Cosine Recall@5,Dim 128 Cosine Recall@10,Dim 128 Cosine Ndcg@10,Dim 128 Cosine Mrr@10,Dim 128 Cosine Map@100,Dim 64 Cosine Accuracy@1,Dim 64 Cosine Accuracy@3,Dim 64 Cosine Accuracy@5,Dim 64 Cosine Accuracy@10,Dim 64 Cosine Precision@1,Dim 64 Cosine Precision@3,Dim 64 Cosine Precision@5,Dim 64 Cosine Precision@10,Dim 64 Cosine Recall@1,Dim 64 Cosine Recall@3,Dim 64 Cosine Recall@5,Dim 64 Cosine Recall@10,Dim 64 Cosine Ndcg@10,Dim 64 Cosine Mrr@10,Dim 64 Cosine Map@100,Sequential Score
1,0.632700,No log,0.870000,0.870000,0.870000,0.920000,0.870000,0.870000,0.870000,0.460000,0.174000,0.522000,0.870000,0.920000,0.897050,0.878333,0.895565,0.860000,0.860000,0.860000,0.930000,0.860000,0.860000,0.860000,0.465000,0.172000,0.516000,0.860000,0.930000,0.897870,0.871667,0.889951,0.860000,0.860000,0.860000,0.920000,0.860000,0.860000,0.860000,0.460000,0.172000,0.516000,0.860000,0.920000,0.892460,0.870000,0.887736,0.860000,0.860000,0.860000,0.930000,0.860000,0.860000,0.860000,0.465000,0.172000,0.516000,0.860000,0.930000,0.897870,0.871667,0.890506,0.810000,0.810000,0.810000,0.910000,0.810000,0.810000,0.810000,0.455000,0.162000,0.486000,0.810000,0.910000,0.864099,0.826667,0.858286,0.864099
2,0.169200,No log,0.840000,0.840000,0.840000,0.930000,0.840000,0.840000,0.840000,0.465000,0.168000,0.504000,0.840000,0.930000,0.888690,0.855000,0.877943,0.880000,0.880000,0.880000,0.930000,0.880000,0.880000,0.880000,0.465000,0.176000,0.528000,0.880000,0.930000,0.907050,0.888333,0.903884,0.870000,0.870000,0.870000,0.920000,0.870000,0.870000,0.870000,0.460000,0.174000,0.522000,0.870000,0.920000,0.897050,0.878333,0.895931,0.860000,0.860000,0.860000,0.950000,0.860000,0.860000,0.860000,0.475000,0.172000,0.516000,0.860000,0.950000,0.908690,0.875000,0.894979,0.840000,0.840000,0.840000,0.930000,0.840000,0.840000,0.840000,0.465000,0.168000,0.504000,0.840000,0.930000,0.888690,0.855000,0.879192,0.888690


TrainOutput(global_step=30, training_loss=0.3415579358736674, metrics={'train_runtime': 153.4004, 'train_samples_per_second': 11.734, 'train_steps_per_second': 0.196, 'total_flos': 0.0, 'train_loss': 0.3415579358736674, 'epoch': 2.0})

In [ ]:
# Save the best model based on our eval_dim_128_cosine_ndcg@10 criteria
trainer1.save_model()

In [ ]:
fine_tuned_model = SentenceTransformer(
    args1.output_dir, device="cuda" if torch.cuda.is_available() else "cpu"
)

# Evaluate the model
ft_results = evaluator(fine_tuned_model)

# Print header
print("Fine Tuned Model Evaluation Results")
print("-" * 85)
print(f"{'Metric':15} {'768d':>12} {'512d':>12} {'256d':>12} {'128d':>12} {'64d':>12}")
print("-" * 85)

# List of metrics to display
metrics = [
    'ndcg@10',
    'mrr@10',
    'accuracy@1',
    'precision@3',
    'precision@5',
    'recall@3',

]
# Print each metric
for metric in metrics:
    values = []
    for dim in matryoshka_dimensions:
        key = f"dim_{dim}_cosine_{metric}"
        values.append(ft_results[key])

    # Highlight NDCG@10
    metric_name = f"=={metric}==" if metric == "ndcg@10" else metric
    print(f"{metric_name:15}", end="  ")
    for val in values:
        print(f"{val:12.4f}", end=" ")
    print()

# Print sequential score
print("-" * 85)
print(f"{'seq_score:'} {ft_results['sequential_score']:1f}")

Fine Tuned Model Evaluation Results
-------------------------------------------------------------------------------------
Metric                  768d         512d         256d         128d          64d
-------------------------------------------------------------------------------------
==ndcg@10==            0.8887       0.9070       0.8970       0.9087       0.8887 
mrr@10                 0.8550       0.8883       0.8783       0.8750       0.8550 
accuracy@1             0.8400       0.8800       0.8700       0.8600       0.8400 
precision@3            0.8400       0.8800       0.8700       0.8600       0.8400 
precision@5            0.8400       0.8800       0.8700       0.8600       0.8400 
recall@3               0.5040       0.5280       0.5220       0.5160       0.5040 
-------------------------------------------------------------------------------------
seq_score: 0.888690


In [ ]:
# Upload model to hub
trainer1.model.push_to_hub("codebert-base-code-embed-mrl-langchain-langgraph")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...u1ecc38/model.safetensors:   0%|          |  551kB /  499MB            

'https://huggingface.co/shubharuidas/codebert-base-code-embed-mrl-langchain-langgraph/commit/f320c0bb290168990530d234e2efa36e666648cc'